# Describe the request body

The `requestBody` field in OpenAPI describes the body of a request that an API client can send to the server, including the content type(s) supported and the schema for the body content.

When the endpoint handler method accepts parameters that are bound to the request body, ASP.NET Core generates a corresponding `requestBody` for the operation in the OpenAPI document. Metadata for the request body can also be specified using attributes or extension methods. Additional metadata can be set with a [document transformer] or [operation transformer].

If the endpoint does not define any parameters bound to the request body, but instead consumes the request body from the [`HttpContext`] directly, ASP.NET Core provides mechanisms to specify request body metadata. This is a common scenario for endpoints that process the request body as a stream.

Some request body metadata can be determined from the [`FromBody`] or [`FromForm`] parameters of the route handler method.

A description for the request body can be set with a [`[Description]`] attribute on the [`FromBody`] or [`FromForm`] parameter.

If the [`FromBody`] parameter is non-nullable and the `EmptyBodyBehavior` is not set to `EmptyBodyBehavior.Allow` in the [`FromBody`] attribute, the request body is required and the `required` field of the `requestBody` is set to `true` in the generated OpenAPI document.
Form bodies are always required and have `required` set to `true`.

Use a [document transformer] or an [operation transformer] to set the `example`, `examples`, or `encoding` fields, or to add specification extensions for the request body in the generated OpenAPI document.

Other mechanisms for setting request body metadata depend on the type of app being developed and are described in the following sections.

[document transformer]: https://learn.microsoft.com/aspnet/core/fundamentals/openapi/aspnetcore-openapi?view=aspnetcore-9.0#use-document-transformers
[operation transformer]: https://learn.microsoft.com/aspnet/core/fundamentals/openapi/aspnetcore-openapi?view=aspnetcore-9.0#use-operation-transformers

[`FromBody`]: https://docs.microsoft.com/dotnet/api/microsoft.aspnetcore.mvc.frombodyattribute
[`FromForm`]: https://docs.microsoft.com/dotnet/api/microsoft.aspnetcore.mvc.fromformattribute
[`HttpContext`]: https://docs.microsoft.com/dotnet/api/microsoft.aspnetcore.http.httpcontext
[`[Description]`]: https://learn.microsoft.com/dotnet/api/system.componentmodel.descriptionattribute


## Minimal APIs

By default, an endpoint that defines a [`FromBody`] parameter will accept `application/json` content-type and an endpoint with one or more [`FromBody`] parameters will accept either `multipart/form-data` or `application/x-www-form-urlencoded`.

Support for these default content types is built in to Minimal APIs, but other content types require custom binding.
See the [Custom binding](https://learn.microsoft.com/aspnet/core/fundamentals/minimal-apis/parameter-binding) topic of the Minimal APIs documentation for more information.

[Custom binding]: https://learn.microsoft.com/aspnet/core/fundamentals/minimal-apis/parameter-binding

There are several ways to specify a different content type for the request body.
If the type of the [`FromBody`] parameter implements [`IEndpointParameterMetadataProvider`], ASP.NET Core uses this interface to determine the content type(s) the request body. The [`Accepts`] extension method can also be used to specify the content type of the request body.

In the following example, the endpoint accepts a `Todo` object in the request body with an expected content-type of `application/xml`.

```csharp
app.MapPut("/todos/{id}", (int id, Todo todo) => ...)
  .Accepts<Todo>("application/xml");
```

The `Todo` class must implement the [`IBindableFromHttpContext<Todo>`] interface to provide a custom binding for the request body. For example

```csharp
public class Todo : IBindableFromHttpContext<Todo>
{
    public static async ValueTask<Todo?> BindAsync(HttpContext context, ParameterInfo parameter)
    {
        var xmlDoc = await XDocument.LoadAsync(context.Request.Body, LoadOptions.None, context.RequestAborted);
        var serializer = new XmlSerializer(typeof(Todo));
        return (Todo?)serializer.Deserialize(xmlDoc.CreateReader());
    }
```

An alternative to the [`Accepts`] extension method is to implement the [`IEndpointParameterMetadataProvider`] interface in the parameter type. The framework uses the [`PopulateMetadata`] method of this interface to set the content type(s) and type of the body content of the request body. For example, the `Todo` class that supports `application/xml` content-type as above can use [`IEndpointParameterMetadataProvider`] to provide this information to the framework.

```csharp
public class Todo : IEndpointParameterMetadataProvider
{
    public static void PopulateMetadata(ParameterInfo parameter, EndpointBuilder builder)
    {
        builder.Metadata.Add(new AcceptsMetadata(["application/xml", "text/xml"], typeof(XmlBody)));
    }
}
```

If the endpoint does not define any parameters bound to the request body, use the [`Accepts`] extension method to specify the content type that the endpoint accepts. 

Note that if you specify [`Accepts`] multiple times, only the last one will be used -- they are not combined.

## Controllers

In controller-based apps, ASP.NET Core uses an [InputFormatter] to deserialize the request body.
InputFormatters are 

When not specified using one of the mechamisms described below, the content type of the request body may be
any content type accepted by an [InputFormatter] for the [FromBody] parameter type.

 so the content entries in the `requestBody` object are based on the input formatters that are
configured for the application. ASP.NET Core MVC includes built-in input formatters for JSON and XML,
though only the JSON input formatter is enabled by default.

A route handler without a [Consumes] filter will accept any content type
for which an available input formatter handles the type of the [FromBody] parameter.
In this case, the requestBody in the generated OpenAPI document will contain all the content types
for input formatters that can handle the [FromBody] parameter type.
The built-in JSON input formatter supports the `application/json`, `text/json`, and `application/*+json` content types.
The built-in XML input formatter supports the `application/xml`, `text/xml`, and `application/*+xml` content types.

You can use a [Consumes] filter, which is configured with the `[Consumes]` attribute, to restrict the content types that a route handler will accept.

A [Consumes] filter cannot add support for a content type
that is not already supported by an input formatter. In this scenario, the requestBody in the
generated OpenAPI document will claim to support "application/json" and _not_ the content type
specified in the Consumes filter. However, a request with either content type will fail
at runtime with a 415 Unsupported Media Type response.

Note that the JSON and XML input formatters can handle the string type, so you must specify
a Consumes filter to restrict the content type to "text/plain" if that is desired.

For content types other than JSON or XML, you need to create a custom input formatter.
For more detailed information and examples, you can refer to the [Custom formatters in ASP.NET Core
Web API](https://learn.microsoft.com/aspnet/core/web-api/advanced/custom-formatters) documentation.

If the route handler does not have a [FromBody] or [FromForm] parameter, the route handler may read
the request body directly from the `Request.Body` stream and may use the Consumes attribute to
restrict the content types allowed, but no requestBody is generated in the OpenAPI document.

### multipart/form-data

An operation that accepts `multipart/form-data` should use `Accepts` to set the correct MIME type and a `FromBody` method parameter with a type that defines the form-data fields.

In [ ]:
curl -s -D - -X POST `
  -H "Content-Type: application/json" `
  -d '{"text": "Hello, world!"}' `
  http://localhost:5110/optional-body

HTTP/1.1 200 OK
Content-Type: application/json; charset=utf-8
Date: Fri, 18 Oct 2024 14:17:35 GMT
Server: Kestrel
Transfer-Encoding: chunked

"Good to go - body present"


In [ ]:
curl -s -D - -X POST `
  http://localhost:5110/optional-body

HTTP/1.1 200 OK
Content-Type: application/json; charset=utf-8
Date: Fri, 18 Oct 2024 14:19:01 GMT
Server: Kestrel
Transfer-Encoding: chunked

"Good to go - no body"


In [ ]:
curl -s -D - -X POST `
  -H "Content-Type: application/json" `
  -d '{"text": "Hello, world!"}' `
  http://localhost:5110/allow-empty-body

HTTP/1.1 200 OK
Content-Type: application/json; charset=utf-8
Date: Fri, 18 Oct 2024 14:41:05 GMT
Server: Kestrel
Transfer-Encoding: chunked

"Good to go - body present"


In [ ]:
curl -s -D - -X POST `
  http://localhost:5110/allow-empty-body

HTTP/1.1 200 OK
Content-Type: application/json; charset=utf-8
Date: Fri, 18 Oct 2024 14:41:17 GMT
Server: Kestrel
Transfer-Encoding: chunked

"Good to go - no body"
